# robot-toolkit Tutorial: Inverse Kinematics

This notebook demonstrates how to use the robot-toolkit library for inverse kinematics (IK) of 6-DOF serial manipulators.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from robot_ik import six_dof_articulated, RobotModel, DHParam

## 1. Load Robot Model

Load the pre-configured 6-DOF articulated manipulator:

In [ ]:
robot = six_dof_articulated()
print(f"Robot DOF: {robot.n}")
print(f"Joint limits: {robot.joint_limits}")

## 2. Forward Kinematics

Compute end-effector pose from joint angles:

In [ ]:
# Joint configuration (radians)
q = np.array([0.0, 0.5, 0.5, 0.0, 0.5, 0.0])

# Compute forward kinematics
T = robot.forward_kinematics(q)
print("End-effector transform:")
print(T)

# Extract position and orientation
position = T[:3, 3]
print(f"\nPosition: {position}")
print(f"Orientation matrix:\n{T[:3, :3]}")

## 3. Inverse Kinematics

Solve for joint angles given a target end-effector pose:

In [ ]:
# Target pose (position + orientation)
target_position = np.array([0.3, 0.2, 0.4])
target_R = np.eye(3)  # Identity orientation

target = np.eye(4)
target[:3, 3] = target_position
target[:3, :3] = target_R

# Solve IK
success, q_solution, iterations, residual = robot.ik_solve(
    target,
    max_iterations=100,
    damping_lambda=0.01,
)

print(f"Success: {success}")
print(f"Iterations: {iterations}")
print(f"Final residual: {residual:.6f}")
print(f"Solution: {q_solution}")

## 4. Verify Solution

Check that the solution reaches the target:

In [ ]:
# Forward kinematics of solution
T_solution = robot.forward_kinematics(q_solution)
pos_error = np.linalg.norm(T_solution[:3, 3] - target[:3, 3])

print(f"Position error: {pos_error*1000:.2f} mm")
print(f"Success criterion: {pos_error < 0.001}")

## 5. Multiple IK Solutions

Try different initial guesses for multiple solutions:

In [ ]:
solutions = []
initial_guesses = [
    np.zeros(6),
    np.ones(6) * 0.5,
    np.array([0.5, -0.5, 0.5, -0.5, 0.5, -0.5]),
]

for q_init in initial_guesses:
    success, q_sol, iters, res = robot.ik_solve(target, q_init=q_init)
    if success:
        # Check if unique solution
        is_unique = True
        for q_prev in solutions:
            if np.linalg.norm(q_sol - q_prev) < 0.01:
                is_unique = False
                break
        if is_unique:
            solutions.append(q_sol)

print(f"Found {len(solutions)} unique solutions")
for i, q_sol in enumerate(solutions):
    print(f"Solution {i+1}: {q_sol}")

## 6. Cartesian Path Following

Generate a straight-line path in Cartesian space and solve IK for each waypoint:

In [ ]:
from robot_ik import cartesian_straight_line

start_q = np.zeros(6)
target_pose = np.eye(4)
target_pose[:3, 3] = np.array([0.2, 0.1, 0.3])

# Generate Cartesian trajectory
traj = cartesian_straight_line(robot, start_q, target_pose, duration=2.0, dt=0.05)

print(f"Trajectory points: {len(traj.time_points)}")
print(f"Duration: {traj.duration} s")
print(f"\nFirst 3 joint configurations:")
for i in range(min(3, len(traj.joint_positions))):
    print(f"  t={traj.time_points[i]:.2f}s: {traj.joint_positions[i]}")

## 7. Visualization

Visualize the robot trajectory:

In [ ]:
from robot_ik import visualize_trajectory

# Visualize trajectory
visualize_trajectory(traj, robot)

## Summary

In this tutorial, you learned:
- How to load a robot model
- Forward kinematics for pose computation
- Inverse kinematics with damped least-squares
- Multiple IK solutions with different initial guesses
- Cartesian path following

## Next Steps

- Explore `tutorial_dynamics.ipynb` for rigid body dynamics
- Explore `tutorial_trajectory_planning.ipynb` for trajectory generation
- Check `examples/` for more demos